# 00 — Data Cleaning & Exploratory Data Analysis

This notebook documents the cleaning pipeline and explores the dataset to
make informed decisions for the four analytical features.

**Rule:** every chart must answer a question, and every question must lead
to a decision. If a plot doesn't change what we do next, it doesn't belong here.



## 0. Setup



In [ ]:
import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", palette="muted", font_scale=0.95)
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

# If running in Colab, mount Drive and set the path:
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT = '/content/drive/MyDrive/amazon-intelligence'
# sys.path.insert(0, PROJECT)

# If running locally:
PROJECT = str(Path.cwd())
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

print(f"Project root: {PROJECT}")



## 1. Load Raw Data



In [ ]:
from analysis.cleaning import load_raw

products_raw, reviews_raw = load_raw()

print(f"Products : {len(products_raw)} rows × {len(products_raw.columns)} columns")
print(f"Reviews  : {len(reviews_raw)} rows × {len(reviews_raw.columns)} columns")
print(f"\nProduct columns: {list(products_raw.columns)}")
print(f"\nReview columns : {list(reviews_raw.columns)}")



## 2. Missing Values

This is the most important section for decision-making. Every null has a
reason, and the reason determines whether we drop, impute, or flag.



In [ ]:
# --- Products: null rates ---
null_rates = products_raw.isna().mean().sort_values(ascending=False)
null_rates_pct = (null_rates * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
fields_to_show = null_rates_pct[null_rates_pct > 0]
if len(fields_to_show) == 0:
    fields_to_show = null_rates_pct.head(10)

fields_to_show.plot(kind="barh", ax=axes[0], color="#2E5C8A")
axes[0].set_xlabel("Missing (%)")
axes[0].set_title("Missing Values — Products")
axes[0].axvline(x=10, color="red", linestyle="--", alpha=0.5, label="10% threshold")
axes[0].legend()

# Heatmap of nulls for key fields
key_fields = ["title", "brand", "price", "rating", "review_count",
              "description", "image_url", "category_breadcrumb"]
key_fields = [f for f in key_fields if f in products_raw.columns]
null_matrix = products_raw[key_fields].isna().astype(int)
sns.heatmap(null_matrix.T, cbar=False, ax=axes[1], cmap="YlOrRd",
            yticklabels=True, xticklabels=False)
axes[1].set_title("Null Pattern (yellow = present, red = missing)")
axes[1].set_xlabel("Products")

plt.tight_layout()
plt.savefig("data/clean/eda_missing_products.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nNull rates (products):")
for field, rate in null_rates_pct.items():
    if rate > 0:
        print(f"  {field:<25} {rate:>5}%")



In [ ]:
# --- Reviews: null rates ---
review_nulls = reviews_raw.isna().mean().sort_values(ascending=False)
review_nulls_pct = (review_nulls * 100).round(1)

fig, ax = plt.subplots(figsize=(10, 4))
review_nulls_pct[review_nulls_pct > 0].plot(kind="barh", ax=ax, color="#B8620B")
ax.set_xlabel("Missing (%)")
ax.set_title("Missing Values — Reviews")
plt.tight_layout()
plt.savefig("data/clean/eda_missing_reviews.png", dpi=150, bbox_inches="tight")
plt.show()

print("\nNull rates (reviews):")
for field, rate in review_nulls_pct.items():
    if rate > 0:
        print(f"  {field:<25} {rate:>5}%")



### Missing Value Decisions

| Field | Missing | Why | Decision |
|---|---|---|---|
| `price` | ~8% | Out-of-stock or variant-only pricing | Flag with `price_status`. Exclude from Feature D labels |
| `brand` | ~3% | Sellers left the byline blank | Keep null for Feature A. Fill "Unknown" for Feature D |
| `description` | <1% | Sparse listings | Use title + bullets instead |
| `review body` | <1% | Rating-only reviews (no text written) | Drop for Feature B |
| `review title` | ~varies | Amazon's new layout sometimes omits it | Use body only for Feature B |
| `helpful_votes` | ~50% | Amazon doesn't show it for all reviews | Null means "not displayed", not zero |

**Key principle:** null rating and null review_count are NOT imputed with
zero. A product with no reviews is information, not a gap.



## 3. Duplicates



In [ ]:
# --- Products ---
asin_dupes = products_raw.duplicated(subset="asin", keep=False).sum()
print(f"Duplicate ASINs: {asin_dupes}")

# --- Reviews ---
review_dedup_cols = [c for c in ["asin", "reviewer_name", "date"]
                     if c in reviews_raw.columns]
review_dupes = reviews_raw.duplicated(subset=review_dedup_cols, keep=False).sum()
print(f"Duplicate reviews (same asin + reviewer + date): {review_dupes}")



## 4. Run the Cleaning Pipeline



In [ ]:
from analysis.cleaning import load_and_clean

products, reviews = load_and_clean(save=True, detect_lang=True)

print(f"\nAfter cleaning:")
print(f"  Products : {len(products)}")
print(f"  Reviews  : {len(reviews)}")



## 5. Price Analysis

This section directly drives Feature D (price tier classification).
The key question: should tiers be global or per-keyword?



In [ ]:
# --- Price distribution ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Overall distribution
products["price"].dropna().plot(kind="hist", bins=50, ax=axes[0],
                                 color="#2E5C8A", edgecolor="white")
axes[0].set_xlabel("Price ($)")
axes[0].set_title("Price Distribution (all products)")
axes[0].axvline(products["price"].median(), color="red", linestyle="--",
                label=f"Median: ${products['price'].median():.0f}")
axes[0].legend()

# Log-scale to see the shape better
products["price"].dropna().plot(kind="hist", bins=50, ax=axes[1],
                                 color="#0F7F6C", edgecolor="white")
axes[1].set_xscale("log")
axes[1].set_xlabel("Price ($ log scale)")
axes[1].set_title("Price Distribution (log scale)")

# Price status
if "price_status" in products.columns:
    products["price_status"].value_counts().plot(kind="bar", ax=axes[2],
                                                  color="#B8620B", edgecolor="white")
    axes[2].set_title("Price Status")
    axes[2].set_ylabel("Count")
    plt.setp(axes[2].get_xticklabels(), rotation=45, ha="right")

plt.tight_layout()
plt.savefig("data/clean/eda_price_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print("Price statistics:")
print(products["price"].describe().round(2).to_string())



In [ ]:
# --- THE CRITICAL PLOT: Price by keyword ---
# If global terciles put every DSLR in "premium" and every phone case in
# "budget", Feature D is a CATEGORY CLASSIFIER wearing a price-tier costume.

fig, ax = plt.subplots(figsize=(14, 6))

keyword_order = (products.groupby("search_keyword")["price"]
                 .median().sort_values().index)

products.boxplot(column="price", by="search_keyword", ax=ax,
                 vert=False, grid=False)

# Add global tercile lines
if products["price"].notna().sum() > 10:
    terciles = products["price"].quantile([1/3, 2/3])
    for q, label in zip(terciles, ["budget|mid", "mid|premium"]):
        ax.axvline(q, color="red", linestyle="--", alpha=0.7, label=f"{label}: ${q:.0f}")
    ax.legend()

ax.set_title("Price by Keyword — why global terciles DON'T work\n"
             "(red lines = global tercile boundaries)")
ax.set_xlabel("Price ($)")
plt.suptitle("")
plt.tight_layout()
plt.savefig("data/clean/eda_price_by_keyword.png", dpi=150, bbox_inches="tight")
plt.show()



### Price Decision for Feature D

**Finding:** Global price terciles track product *category*, not price *tier*.
A $40 keyboard and a $40 DSLR camera are not the same tier.

**Decision:** Bin prices into budget / mid-range / premium **within each
keyword group** using `pd.qcut`. This way "budget DSLR" competes against
other DSLRs, not against phone cases.



## 6. Rating and Review Count



In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Rating distribution
products["rating"].dropna().plot(kind="hist", bins=20, ax=axes[0],
                                  color="#2E5C8A", edgecolor="white")
axes[0].set_xlabel("Rating")
axes[0].set_title("Product Rating Distribution")
axes[0].axvline(products["rating"].median(), color="red", linestyle="--",
                label=f"Median: {products['rating'].median():.1f}")
axes[0].legend()

# Review count (log scale)
if "review_count" in products.columns:
    products["review_count"].dropna().plot(kind="hist", bins=50, ax=axes[1],
                                            color="#0F7F6C", edgecolor="white")
    axes[1].set_xscale("log")
    axes[1].set_xlabel("Review Count (log scale)")
    axes[1].set_title("Review Count Distribution")

# Review count vs rating
if "review_count" in products.columns:
    ax2 = axes[2]
    valid = products.dropna(subset=["rating", "review_count"])
    ax2.scatter(valid["review_count"], valid["rating"], alpha=0.3, s=10, color="#5B4B8A")
    ax2.set_xscale("log")
    ax2.set_xlabel("Review Count (log)")
    ax2.set_ylabel("Rating")
    ax2.set_title("Rating vs Review Count")

plt.tight_layout()
plt.savefig("data/clean/eda_ratings.png", dpi=150, bbox_inches="tight")
plt.show()

print("Review count statistics:")
print(products["review_count"].describe().round(0).to_string())
print(f"\nMedian: {products['review_count'].median():.0f}")
print(f"Mean:   {products['review_count'].mean():.0f}")
print(f"Max:    {products['review_count'].max():.0f}")



### Rating & Review Count Decision

**Finding:** `review_count` spans several orders of magnitude
(median ~X vs max ~Y). Heavily right-skewed.

**Decision:** Use `log1p(review_count)` as the feature in Feature D.
Raw counts would dominate any linear or distance-based model.

**Finding:** Rating and review_count show little correlation —
popular products are not necessarily higher-rated.



## 7. Review Analysis (drives Feature B)



In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Star rating distribution
if "rating" in reviews.columns:
    reviews["rating"].value_counts().sort_index().plot(
        kind="bar", ax=axes[0, 0], color="#2E5C8A", edgecolor="white")
    axes[0, 0].set_title("Review Star Distribution")
    axes[0, 0].set_xlabel("Stars")
    axes[0, 0].set_ylabel("Count")

    # Print the skew
    total = len(reviews)
    for star in sorted(reviews["rating"].dropna().unique()):
        count = (reviews["rating"] == star).sum()
        pct = count / total * 100
        print(f"  {star:.0f} star: {count:>5} ({pct:.1f}%)")

# Star-based sentiment distribution
if "star_sentiment" in reviews.columns:
    reviews["star_sentiment"].value_counts().plot(
        kind="bar", ax=axes[0, 1], color=["#0F7F6C", "#B8620B", "#2E5C8A"],
        edgecolor="white")
    axes[0, 1].set_title("Weak Labels from Stars")
    axes[0, 1].set_xlabel("Sentiment")
    plt.setp(axes[0, 1].get_xticklabels(), rotation=0)

# Review body length
if "body_clean" in reviews.columns:
    lengths = reviews["body_clean"].dropna().str.split().str.len()
    lengths.plot(kind="hist", bins=50, ax=axes[1, 0], color="#5B4B8A", edgecolor="white")
    axes[1, 0].set_xlabel("Word Count")
    axes[1, 0].set_title("Review Body Length")
    axes[1, 0].axvline(lengths.median(), color="red", linestyle="--",
                       label=f"Median: {lengths.median():.0f} words")
    axes[1, 0].legend()

# Reviews per product
if "asin" in reviews.columns:
    rev_per_product = reviews.groupby("asin").size()
    rev_per_product.plot(kind="hist", bins=30, ax=axes[1, 1],
                          color="#B8620B", edgecolor="white")
    axes[1, 1].set_xlabel("Reviews per Product")
    axes[1, 1].set_title(f"Reviews per Product (median: {rev_per_product.median():.0f})")

plt.tight_layout()
plt.savefig("data/clean/eda_reviews.png", dpi=150, bbox_inches="tight")
plt.show()



### Review Decisions for Feature B

**Finding:** Heavy positive skew — ~60-70% are 5-star.

**Decision:** Use **macro-F1**, not accuracy. Accuracy would score high by
predicting "positive" for everything. Use `class_weight='balanced'` in all
classifiers. Neutral will be the weakest class.

**Finding:** Short reviews (under 3 words) carry no useful signal.

**Decision:** Already dropped in the cleaning step.



## 8. Per-Keyword Breakdown



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Products per keyword
kw_counts = products["search_keyword"].value_counts().sort_values()
kw_counts.plot(kind="barh", ax=axes[0], color="#2E5C8A")
axes[0].set_xlabel("Products")
axes[0].set_title("Products per Keyword")

# Reviews per keyword
if "search_keyword" in reviews.columns:
    rev_kw = reviews["search_keyword"].value_counts().sort_values()
    rev_kw.plot(kind="barh", ax=axes[1], color="#0F7F6C")
    axes[1].set_xlabel("Reviews")
    axes[1].set_title("Reviews per Keyword")

plt.tight_layout()
plt.savefig("data/clean/eda_per_keyword.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Products per keyword: min={kw_counts.min()}, max={kw_counts.max()}, "
      f"median={kw_counts.median():.0f}")



In [ ]:
# --- Null rates by keyword (the diagnostic that finds broken selectors) ---
key_fields = [f for f in ["title", "brand", "price", "rating", "review_count",
                          "description", "image_url"]
              if f in products.columns]

null_by_kw = products.groupby("search_keyword")[key_fields].apply(
    lambda g: g.isna().mean()
).round(2)

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(null_by_kw, annot=True, fmt=".0%", cmap="YlOrRd", ax=ax,
            linewidths=0.5, cbar_kws={"label": "Missing Rate"})
ax.set_title("Missing Rate by Keyword\n(red spots = unhandled layout variant OR genuinely absent)")
plt.tight_layout()
plt.savefig("data/clean/eda_null_by_keyword.png", dpi=150, bbox_inches="tight")
plt.show()



### Per-Keyword Decision

If a field is null **only for certain keywords**, it means an unhandled
Amazon page layout — fix the selector in `scraper/selectors.py`.

If it's null **uniformly**, it's genuinely absent — document and move on.



## 9. Brand Analysis (drives Features A and D)



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Top brands
if "brand" in products.columns:
    top_brands = products["brand"].value_counts().head(20)
    top_brands.plot(kind="barh", ax=axes[0], color="#2E5C8A")
    axes[0].set_xlabel("Count")
    axes[0].set_title("Top 20 Brands")

    brand_count = products["brand"].nunique()
    print(f"Unique brands: {brand_count}")
    print(f"Products without brand: {products['brand'].isna().sum()}")
    print(f"\nTop 10:")
    print(top_brands.head(10).to_string())

# Brand vs price
valid = products.dropna(subset=["brand", "price"])
if len(valid) > 0:
    top_brand_names = valid["brand"].value_counts().head(15).index
    brand_data = valid[valid["brand"].isin(top_brand_names)]
    brand_data.boxplot(column="price", by="brand", ax=axes[1],
                       vert=False, grid=False)
    axes[1].set_title("Price by Brand (top 15)")
    axes[1].set_xlabel("Price ($)")
    plt.suptitle("")

plt.tight_layout()
plt.savefig("data/clean/eda_brands.png", dpi=150, bbox_inches="tight")
plt.show()



### Brand Decision

**For Feature A (similarity):** Brand match is a useful similarity signal.
Keep as-is.

**For Feature D (price tier):** High cardinality — use one-hot for the top
15-20 brands, group the rest as "other". Or use target encoding if brands
are too sparse.



## 10. Text Field Analysis (drives Features A and D)



In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Title length
if "title" in products.columns:
    title_words = products["title"].dropna().str.split().str.len()
    title_words.plot(kind="hist", bins=30, ax=axes[0], color="#2E5C8A", edgecolor="white")
    axes[0].set_xlabel("Word Count")
    axes[0].set_title("Title Length")

# Description length
if "description" in products.columns:
    desc_words = products["description"].dropna().str.split().str.len()
    desc_words.plot(kind="hist", bins=30, ax=axes[1], color="#0F7F6C", edgecolor="white")
    axes[1].set_xlabel("Word Count")
    axes[1].set_title("Description Length")

# Combined text length
if "combined_text" in products.columns:
    combined_words = products["combined_text"].str.split().str.len()
    combined_words.plot(kind="hist", bins=30, ax=axes[2], color="#5B4B8A", edgecolor="white")
    axes[2].set_xlabel("Word Count")
    axes[2].set_title("Combined Text Length (title+brand+desc+bullets)")

plt.tight_layout()
plt.savefig("data/clean/eda_text_lengths.png", dpi=150, bbox_inches="tight")
plt.show()



In [ ]:
# --- LEAKAGE CHECK: do any titles contain price strings? ---
import re
price_in_text = products["title"].fillna("").apply(
    lambda t: bool(re.search(r"[\$£€]\s?\d[\d,.]*", t))
).sum()

desc_price = 0
if "description" in products.columns:
    desc_price = products["description"].fillna("").apply(
        lambda t: bool(re.search(r"[\$£€]\s?\d[\d,.]*", t))
    ).sum()

print(f"LEAKAGE CHECK for Feature D:")
print(f"  Titles with currency strings    : {price_in_text}")
print(f"  Descriptions with currency      : {desc_price}")
if price_in_text > 0 or desc_price > 0:
    print(f"  ACTION: scrub these before vectorising for Feature D")
else:
    print(f"  No leakage found in text fields")



### Text Decision for Feature D

**Finding:** Currency strings found in X titles / Y descriptions.

**Decision:** Regex-scrub `$XX.XX` patterns from text before TF-IDF
vectorisation for Feature D. Without this, the model learns to read the
price from the title, which defeats the purpose.



## 11. Image and Video Coverage (drives Feature C)



In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Image status
if "image_url" in products.columns:
    img_present = products["image_url"].notna().sum()
    img_missing = products["image_url"].isna().sum()
    axes[0].bar(["Has image", "No image"], [img_present, img_missing],
                color=["#0F7F6C", "#B8620B"])
    axes[0].set_title("Image Coverage")
    axes[0].set_ylabel("Products")
    print(f"Images: {img_present} present, {img_missing} missing")

# Video status
if "has_video" in products.columns:
    vid_counts = products["has_video"].value_counts()
    has_url = products["video_url"].notna().sum()
    labels = ["Has video + URL", "Has video, no URL", "No video"]
    values = [has_url,
              int(products["has_video"].sum()) - has_url,
              int((~products["has_video"]).sum())]
    axes[1].bar(labels, values, color=["#0F7F6C", "#B8620B", "#4A5A6E"])
    axes[1].set_title("Video Coverage")
    axes[1].set_ylabel("Products")
    plt.setp(axes[1].get_xticklabels(), rotation=15, ha="right")
    print(f"Video: {values[0]} with URL, {values[1]} video but no URL, {values[2]} none")

plt.tight_layout()
plt.savefig("data/clean/eda_media_coverage.png", dpi=150, bbox_inches="tight")
plt.show()



## 12. Correlation Matrix (numeric fields)



In [ ]:
numeric_cols = ["price", "rating", "review_count", "reviews_collected",
                "title_word_count", "log_review_count"]
numeric_cols = [c for c in numeric_cols if c in products.columns]

if len(numeric_cols) >= 3:
    corr = products[numeric_cols].corr()
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
                ax=ax, square=True, linewidths=0.5)
    ax.set_title("Correlation Matrix — Numeric Fields")
    plt.tight_layout()
    plt.savefig("data/clean/eda_correlation.png", dpi=150, bbox_inches="tight")
    plt.show()



## 13. Summary of Decisions

| # | Finding | Evidence | Decision | Affects |
|---|---|---|---|---|
| 1 | ~60-70% of reviews are 5-star | Star distribution | macro-F1, not accuracy; `class_weight='balanced'` | Feature B |
| 2 | `review_count` spans 4+ orders of magnitude | Histogram | `log1p` transform | Feature D |
| 3 | Global price terciles track category | Price-by-keyword boxplot | Bin within keyword group | Feature D |
| 4 | Currency strings in some titles | Regex search | Scrub before vectorising | Feature D |
| 5 | ~8% price missing (out-of-stock) | Null rates + `price_status` | Exclude from Feature D labels | Feature D |
| 6 | Brand has moderate cardinality | Brand value counts | Top-20 one-hot for Feature D | Features A, D |
| 7 | Short reviews (<3 words) | Length histogram | Already dropped in cleaning | Feature B |
| 8 | Null rates uniform across keywords | Heatmap | No broken selectors — genuine absence | All |
| 9 | Helpful votes ~50% null | Null rates | Null = not displayed, distinct from 0 | Feature B |
| 10 | Products without reviews exist | Count | Keep in catalogue, exclude from Feature B | Features B, D |

**Update the numbers above** after running on your real data.



## 14. What Each Feature Notebook Should Load

```python
from analysis.cleaning import load_clean
products, reviews = load_clean()
```

| Feature | Primary table | Key columns |
|---|---|---|
| A — Similar products | `products` | `combined_text`, `brand`, `price`, `asin` |
| B — Sentiment | `reviews` | `body_clean`, `rating`, `star_sentiment`, `asin` |
| C — Image grouping | `products` + `data/images/` | `asin`, `image_url`, `search_keyword` |
| D — Price tier | `products` | `combined_text`, `brand`, `rating`, `log_review_count`, `price`, `search_keyword` |



---
*Notebook complete. All charts saved to `data/clean/eda_*.png`.*


In [ ]:
# Display the first few rows of the cleaned data
display(products.head(3))
display(reviews.head(3))


## Raw Data Language Distribution
Let's check the language distribution of the completely raw dataset before any cleaning is applied.

In [ ]:
import sys
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
tqdm.pandas()

# 1. Connect to Project (Uncomment for Colab)
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT = '/content/drive/MyDrive/Amazon Mini Projects/Amazon-Marketplace-Product-Intelligence'
# sys.path.insert(0, PROJECT)

# 2. Load the completely RAW dataset
from analysis.cleaning import load_raw
print("Loading raw data from products.jsonl...")
products_raw, reviews_raw = load_raw()

print(f"Total Raw Reviews: {len(reviews_raw)}")

# 3. Define a safe language detection function
def detect_lang_safe(text):
    try:
        if pd.isna(text) or len(str(text).strip()) < 3:
            return 'unknown'
        from langdetect import detect
        return detect(str(text))
    except:
        return 'unknown'

# 4. Apply Language Detection to the raw reviews
print("\nDetecting languages in raw data (this may take a minute)...")
reviews_raw['raw_language'] = reviews_raw['body'].progress_apply(detect_lang_safe)

# 5. Display Results
print("\n--- Language Distribution in RAW Dataset ---")
language_counts = reviews_raw['raw_language'].value_counts()
print(language_counts)

# 6. Visualize
plt.figure(figsize=(10, 5))
sns.countplot(data=reviews_raw, y='raw_language', hue='raw_language', order=language_counts.index, palette='viridis', legend=False)
plt.title("Distribution of Languages in RAW Reviews", fontsize=14, fontweight='bold')
plt.xlabel("Number of Reviews")
plt.ylabel("Language Code")
plt.show()


## Raw Data Language Distribution
Let's check the language distribution of the completely raw dataset before any cleaning is applied.

In [ ]:
import sys
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
tqdm.pandas()

# 1. Connect to Project (Uncomment for Colab)
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT = '/content/drive/MyDrive/Amazon Mini Projects/Amazon-Marketplace-Product-Intelligence'
# sys.path.insert(0, PROJECT)

# 2. Load the completely RAW dataset
from analysis.cleaning import load_raw
print("Loading raw data from products.jsonl...")
products_raw, reviews_raw = load_raw()

print(f"Total Raw Reviews: {len(reviews_raw)}")

# 3. Define a safe language detection function
def detect_lang_safe(text):
    try:
        if pd.isna(text) or len(str(text).strip()) < 3:
            return 'unknown'
        from langdetect import detect
        return detect(str(text))
    except:
        return 'unknown'

# 4. Apply Language Detection to the raw reviews
print("\nDetecting languages in raw data (this may take a minute)...")
reviews_raw['raw_language'] = reviews_raw['body'].progress_apply(detect_lang_safe)

# 5. Display Results
print("\n--- Language Distribution in RAW Dataset ---")
language_counts = reviews_raw['raw_language'].value_counts()
print(language_counts)

# 6. Visualize
plt.figure(figsize=(10, 5))
sns.countplot(data=reviews_raw, y='raw_language', hue='raw_language', order=language_counts.index, palette='viridis', legend=False)
plt.title("Distribution of Languages in RAW Reviews", fontsize=14, fontweight='bold')
plt.xlabel("Number of Reviews")
plt.ylabel("Language Code")
plt.show()
